# 06 - Morphotope Generation (Spatially-Constrained Ward Clustering)

Following source `generate_clusters.py` — the core HiMoC method:
1. Split tessellation graph into connected components
2. Per component: `QuantileTransformer(uniform)` on primary chars
3. `AgglomerativeClustering(ward)` with tessellation graph as `connectivity`
4. Extract clusters from linkage tree via `fast_hdbscan` (leaf extraction)
5. `min_cluster_size=75`

This is NOT UMAP+HDBSCAN. The spatial constraint comes from the tessellation
contiguity graph passed as the `connectivity` parameter.

In [1]:
import os
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import fcluster
from fast_hdbscan.cluster_trees import (
    condense_tree, cluster_tree_from_condensed_tree,
    extract_leaves, get_cluster_label_vector,
)
from libpysal.graph import read_parquet
from pathlib import Path

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")
PRIMARY_CHARS_DIR = Path(OUTPUT_ROOT) / "05_primary_chars"
GRAPH_DIR = Path(OUTPUT_ROOT) / "03_graphs"
OUT = Path(OUTPUT_ROOT) / "06_morphotopes"
OUT.mkdir(parents=True, exist_ok=True)

Working dir: C:\urban-form\urban_taxonomy
[pipeline_config] City config: config\canada.env


## Load data

In [2]:
X_train = pd.read_parquet(PRIMARY_CHARS_DIR / "primary_chars.parquet")
graph = read_parquet(GRAPH_DIR / "tessellation_graph.parquet")

print(f"Primary chars: {X_train.shape}")
print(f"Tessellation graph: {graph.n} nodes")

Primary chars: (93234, 75)
Tessellation graph: 93234 nodes


## Clustering parameters (from source)

In [3]:
MIN_CLUSTER_SIZE = int(os.environ.get("MIN_CLUSTER_SIZE", "75"))  # source default

# Columns to drop before clustering (source to_drop list)
TO_DROP = [
    'stcSAl', 'stbOri', 'stcOri', 'stbCeA',
    'ldkAre', 'ldkPer', 'lskCCo', 'lskERI',
    'lskCWA', 'ltkOri', 'ltkWNB', 'likWBB', 'likWCe',
    'licBAD', 'misBAD', 'ssbCCM', 'ssbCCD',
]

SCALAR = QuantileTransformer(subsample=None, output_distribution='uniform')
LINKAGE = 'ward'
METRIC = 'euclidean'
EOM_CLUSTERS = False  # source uses leaf extraction, not EOM

## Helper functions (from source `generate_clusters.py`)

In [4]:
def get_linkage_matrix(model):
    """Extract linkage matrix from sklearn AgglomerativeClustering."""
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        count = 0
        for child in merge:
            count += 1 if child < n_samples else counts[child - n_samples]
        counts[i] = count
    return np.column_stack([model.children_, model.distances_, counts]).astype(float)


def preprocess_clustering_data(X, scalar, to_drop):
    """Preprocess: drop non-buildings, drop columns, scale, handle NaN."""
    X = X[X.index >= 0].copy()
    all_drop = []
    for c in to_drop:
        all_drop += X.columns[X.columns.str.contains(c)].tolist()
    X = X.drop(columns=[c for c in all_drop if c in X.columns], errors='ignore')

    vals = scalar.fit_transform(X)
    X = pd.DataFrame(np.nan_to_num(vals), columns=X.columns, index=X.index)

    # Drop zero-variance columns
    stats = X.describe()
    X = X.drop(columns=stats.columns[stats.loc['std'] == 0], errors='ignore')
    return X


def get_clusters(linkage_matrix, min_cluster_size, n_samples, eom=False):
    """Extract clusters from linkage via fast_hdbscan condensed tree."""
    condensed = condense_tree(linkage_matrix, min_cluster_size=min_cluster_size)
    cluster_tree = cluster_tree_from_condensed_tree(condensed)
    selected = extract_leaves(condensed, allow_single_cluster=False)
    return get_cluster_label_vector(condensed, selected, 0, n_samples)

## Run morphotope clustering

Per-component Ward clustering with tessellation graph connectivity.

In [5]:
building_graph = graph.subgraph(graph.unique_ids[graph.unique_ids >= 0])
labels = building_graph.component_labels
results = {}

for label, group in labels.groupby(labels):
    if group.shape[0] <= MIN_CLUSTER_SIZE:
        results[label] = np.full(group.shape[0], -1)
        continue

    # Preprocess
    comp_data = preprocess_clustering_data(
        X_train.loc[group.index.values], scalar=SCALAR, to_drop=TO_DROP
    )
    comp_graph = building_graph.subgraph(group.index.values)

    # Ward with spatial connectivity
    clusterer = AgglomerativeClustering(
        linkage=LINKAGE, metric=METRIC,
        connectivity=comp_graph.transform('B').sparse,
        compute_full_tree=True, compute_distances=True,
    )
    model = clusterer.fit(comp_data)
    ward_tree = get_linkage_matrix(model)

    # Fix non-monotonic distances
    problem_idxs = np.where(ward_tree[1:, 2] < ward_tree[0:-1, 2])[0]
    while problem_idxs.shape[0]:
        ward_tree[problem_idxs + 1, 2] = ward_tree[problem_idxs, 2] + 0.01
        problem_idxs = np.where(ward_tree[1:, 2] < ward_tree[0:-1, 2])[0]

    # Extract clusters
    comp_clusters = get_clusters(ward_tree, MIN_CLUSTER_SIZE, comp_data.shape[0])
    results[label] = comp_clusters

    if len(results) % 10 == 0:
        print(f"  Processed {len(results)} components...")

print(f"Processed {len(results)} connected components")

  Processed 80 components...


  Processed 230 components...


  Processed 450 components...


  Processed 800 components...


Processed 12362 connected components


In [6]:
# Relabel local clusters to global unique labels
label_groups = labels.groupby(labels)
region_cluster_labels = []
for label, comp_clusters in results.items():
    group = label_groups.get_group(label)
    comp_labels = str(label) + '_' + pd.Series(
        comp_clusters.astype(str), index=group.index.values
    )
    region_cluster_labels.append(comp_labels)

region_cluster_labels = pd.concat(region_cluster_labels).sort_index()

n_morphotopes = region_cluster_labels.nunique()
n_noise = (region_cluster_labels.str.endswith('_-1')).sum()
print(f"{n_morphotopes} unique morphotopes, {n_noise:,} noise cells ({100*n_noise/len(region_cluster_labels):.1f}%)")

12568 unique morphotopes, 51,885 noise cells (61.7%)


In [7]:
region_cluster_labels.to_frame('morphotope_label').to_parquet(
    OUT / 'tessellation_labels_morphotopes.parquet'
)
print(f"Saved morphotope labels to {OUT}")

Saved morphotope labels to outputs_canada\regions\4810\06_morphotopes


## Generate morphotope-level aggregated data

In [8]:
def percentile(n):
    def fn(x): return np.percentile(x, n)
    fn.__name__ = f'percentile_{n}'
    return fn

component_data = X_train.loc[region_cluster_labels.index]
component_data = component_data.groupby(region_cluster_labels.values).agg(
    [percentile(25), 'median', percentile(75), 'std', 'mean']
)
component_data[('Size', 'Size')] = (
    X_train.loc[region_cluster_labels.index]
    .groupby(region_cluster_labels.values).size()
)

component_data.to_parquet(OUT / 'data_morphotopes.parquet')
print(f"Saved morphotope data: {component_data.shape}")

Saved morphotope data: (12568, 376)
